## Import

In [33]:
import pandas as pd
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

## Tables

In [34]:
SILVER_COMMUNICATIONS = "../../data/silver/communications"
SILVER_CARRIERS       = "../../data/silver/carriers"
SILVER_BROKERS        = "../../data/silver/brokers"

## Dataframes

In [35]:
df_communications = pd.read_parquet(SILVER_COMMUNICATIONS)

df_carrier        = (
    pd.read_parquet(SILVER_CARRIERS)
    .rename(
        columns={
            "id": "carrier_company_id",
            "name": "carrier_name"
        }
    )
)

df_broker         = (
    pd.read_parquet(SILVER_BROKERS)
    .rename(
        columns={
            "id": "broker_company_id",
            "name": "broker_name"
        }
    )
)

In [36]:
df = (
    df_communications
    .merge(df_carrier, on="carrier_company_id", how="left")
    .merge(df_broker, on="broker_company_id", how="left")
    [
        [
            "external_id",
            "carrier_company_id",
            "carrier_name",
            "broker_company_id",
            "broker_name",
            "direction",
            "channel",
            "status",
            "created_at",
            "updated_at",
            "from_contact_type",
            "to_contact_type",
            "thread_id"
        ]
    ]
)

In [37]:
load_dotenv("../../.env")


database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(database_url)

# with engine.connect() as connection:

#     database = connection.execute(
#         text("SELECT current_database()")
#     ).scalar()

#     print(database)

In [38]:
df_gold = df.head(10)

df_gold.to_sql(
    name="carrier_risk",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

-1